In [26]:
import numpy as np
import pandas as pd
# Dataset
from sklearn.datasets import load_wine

# Models
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV

from sklearn.metrics import f1_score, mean_squared_error



In [27]:
# Loading the dataset
wine = load_wine()

In [28]:
# selecting the features
X = wine.data

#selecting the target
y = wine.target

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

Feature shape: (178, 13)
Target shape: (178,)


In [29]:
# Train test splitting to 80-20
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [30]:
print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])


Training samples: 142
Testing samples: 36


# Decision Tree

In [31]:
#creating the model
decision_tree_clf = DecisionTreeClassifier(random_state=42)

# training the model with our wine dataset
decision_tree_clf.fit(X_train, y_train)

DecisionTreeClassifier(random_state=42)

In [32]:
# making the predictions from the decision tree model
y_pred_decision_tree = decision_tree_clf.predict(X_test)

In [33]:
#calculating f1 score of the decision tree
decision_tree_f1_score = f1_score(y_test, y_pred_decision_tree, average="weighted")
print("Decision Tree F1 Score:", decision_tree_f1_score)

Decision Tree F1 Score: 0.9439974457215836


# Random Forest

In [34]:
# Creating model for random forest
random_forest_clf = RandomForestClassifier(random_state=42)

In [35]:
# Training random forest model
random_forest_clf.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [36]:
# making the predictions from the random forest model
y_pred_random_forest = random_forest_clf.predict(X_test)

In [37]:
random_forest_f1_score = f1_score(y_test, y_pred_random_forest, average="weighted")
print("Random Forest F1 Score:", random_forest_f1_score)


Random Forest F1 Score: 1.0


## Comparison of Models Based on F1 Score

The Decision Tree Classifier achieved a good F1 score, which shows that it can classify the wine dataset reasonably well. However, since it is based on a single decision tree, its performance can be affected by overfitting, especially when the model learns patterns that are too specific to the training data.

The Random Forest Classifier achieved a higher F1 score compared to the Decision Tree Classifier. This model uses an ensemble of multiple decision trees, where each tree makes its own prediction and the final output is decided by combining all the predictions.

Due to this ensemble approach, Random Forest is more robust and less sensitive to noise in the data. As a result, it provides more accurate and reliable predictions, which is reflected in its higher F1 score.


In [38]:
param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 5, 10],
    "min_samples_split": [2, 5, 10]
}

In [39]:
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    scoring="f1_weighted",
    cv=5
)

grid_search.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=RandomForestClassifier(random_state=42),
             param_grid={'max_depth': [None, 5, 10],
                         'min_samples_split': [2, 5, 10],
                         'n_estimators': [50, 100, 200]},
             scoring='f1_weighted')

In [40]:
print("Best Hyperparameters:", grid_search.best_params_)
print("Best F1 Score:", grid_search.best_score_)


Best Hyperparameters: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 100}
Best F1 Score: 0.9782952128219708


In [41]:
# Target = alcohol content
y_reg = X[:, 0]  # alcohol

# Features = all other columns
X_reg = X[:, 1:]

# Split into training and testing
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

print("Training samples:", X_train_r.shape[0])
print("Testing samples:", X_test_r.shape[0])


Training samples: 142
Testing samples: 36


In [42]:
decision_tree_reg = DecisionTreeRegressor(random_state=42)

In [43]:
decision_tree_reg.fit(X_train_r, y_train_r)

DecisionTreeRegressor(random_state=42)

In [47]:
y_pred_decision_tree_reg = decision_tree_reg.predict(X_test_r)

In [48]:
decision_tree_mse = mean_squared_error(y_test_r, y_pred_decision_tree_reg)
print("Decision Tree Regressor MSE:", decision_tree_mse)

Decision Tree Regressor MSE: 0.31197222222222226


In [50]:
random_forest_reg = RandomForestRegressor(random_state=42)

random_forest_reg.fit(X_train_r, y_train_r)

y_pred_random_forest_reg = random_forest_reg.predict(X_test_r)

rf_mse = mean_squared_error(y_test_r, y_pred_random_forest_reg)

print("Random Forest Regressor MSE:", rf_mse)


Random Forest Regressor MSE: 0.15426672999999946


In [51]:
param_dist = {
    "n_estimators": [50, 100, 200, 300],
    "max_depth": [None, 5, 10, 20],
    "min_samples_leaf": [1, 2, 4]
}

In [52]:
random_search = RandomizedSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_distributions=param_dist,
    n_iter=10,                     # number of random combinations to try
    scoring="neg_mean_squared_error",
    cv=5,                          # 5-fold cross-validation
    random_state=42
)

In [53]:
random_search.fit(X_train_r, y_train_r)

RandomizedSearchCV(cv=5, estimator=RandomForestRegressor(random_state=42),
                   param_distributions={'max_depth': [None, 5, 10, 20],
                                        'min_samples_leaf': [1, 2, 4],
                                        'n_estimators': [50, 100, 200, 300]},
                   random_state=42, scoring='neg_mean_squared_error')

In [54]:
print("Best Hyperparameters (Regression):", random_search.best_params_)
print("Best MSE (CV):", -random_search.best_score_)

Best Hyperparameters (Regression): {'n_estimators': 200, 'min_samples_leaf': 1, 'max_depth': 10}
Best MSE (CV): 0.3195298451116163
